# FP-Growth i asocijacijska pravila

U ovom dijelu projekta primjenjuje se algoritam FP-Growth za pronalaženje frekventnih skupova objekata te generiranje asocijacijskih pravila.

Cilj analize je otkriti koje se karakteristike Airbnb smještaja često pojavljuju zajedno te identificirati obrasce koji mogu objasniti visoku cijenu, popularnost i kvalitetu smještaja.

Za analizu se koristi skup podataka obogaćen novim atributima kreiranim tijekom faze inženjerstva atributa.

## Učitavanje podataka i biblioteka

Prvo se učitavaju potrebne Python biblioteke te skup podataka koji sadrži originalne i novoizvedene atribute korištene u daljnjoj analizi.

In [1]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from mlxtend.frequent_patterns import fpgrowth, association_rules

In [5]:
columns_needed = [
    "room_type",
    "neighbourhood_group",
    "cancellation_policy",
    "review_rate_number",
    "availability_ratio",
    "popularity_score",
    "distance_to_center",
    "high_price",
    "professional_host",
    "luxury_listing"
]

df = pd.read_csv(
    "Airbnb_Association_Rules/airbnb_features.csv",
    usecols=columns_needed,
    low_memory=False
)

df.head()

,neighbourhood_group,cancellation_policy,room_type,review_rate_number,availability_ratio,professional_host,distance_to_center,popularity_score,luxury_listing,high_price
0,Brooklyn,strict,Private room,4.0,0.783562,1,12.337898,-0.469815,0,1
1,Manhattan,moderate,Entire home/apt,4.0,0.624658,0,0.508366,0.321559,0,0
2,Brooklyn,moderate,Entire home/apt,4.0,0.882192,0,8.387034,7.147646,0,0
3,Brooklyn,moderate,Private room,5.0,0.613699,0,8.290747,1.198536,0,0
4,Manhattan,flexible,Entire home/apt,3.0,0.002740,0,4.971807,2.300431,0,0


## Pregled podataka

Prije primjene algoritma potrebno je provjeriti strukturu skupa podataka, tipove atributa te eventualno postojanje nedostajućih vrijednosti.

In [6]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 51045 entries, 0 to 51044
Data columns (total 10 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   neighbourhood_group  51045 non-null  str    
 1   cancellation_policy  51045 non-null  str    
 2   room_type            51045 non-null  str    
 3   review_rate_number   51045 non-null  float64
 4   availability_ratio   51045 non-null  float64
 5   professional_host    51045 non-null  int64  
 6   distance_to_center   51045 non-null  float64
 7   popularity_score     51045 non-null  float64
 8   luxury_listing       51045 non-null  int64  
 9   high_price           51045 non-null  int64  
dtypes: float64(4), int64(3), str(3)
memory usage: 3.9 MB


In [7]:
df.isnull().sum()

neighbourhood_group    0
cancellation_policy    0
room_type              0
review_rate_number     0
availability_ratio     0
professional_host      0
distance_to_center     0
popularity_score       0
luxury_listing         0
high_price             0
dtype: int64

## Priprema atributa za asocijacijsku analizu

FP-Growth algoritam radi nad transakcijskim podacima koji se sastoje od prisutnosti ili odsutnosti određenih obilježja.

Zbog toga se kontinuirani atributi pretvaraju u binarne kategorije kako bi se mogli koristiti u analizi asocijacijskih pravila.

In [8]:
df["highly_rated"] = (
    df["review_rate_number"] >= 4
).astype(int)

df["high_availability"] = (
    df["availability_ratio"] > df["availability_ratio"].median()
).astype(int)

df["popular_listing"] = (
    df["popularity_score"] > df["popularity_score"].median()
).astype(int)

df["close_to_center"] = (
    df["distance_to_center"] < df["distance_to_center"].median()
).astype(int)

In [9]:
df[
    [
        "high_price",
        "highly_rated",
        "high_availability",
        "popular_listing",
        "close_to_center",
        "professional_host",
        "luxury_listing"
    ]
].head()

,high_price,highly_rated,high_availability,popular_listing,close_to_center,professional_host,luxury_listing
0,1,1,1,0,0,1,0
1,0,1,1,1,1,0,0
2,0,1,1,1,0,0,0
3,0,1,1,1,0,0,0
4,0,0,0,1,1,0,0


## Formiranje transakcijske tablice

Svaki Airbnb oglas promatra se kao jedna transakcija, dok pojedina svojstva oglasa predstavljaju stavke unutar transakcije.

Za potrebe FP-Growth algoritma formira se binarna tablica u kojoj vrijednost True označava prisutnost određenog obilježja.

In [10]:
basket = pd.DataFrame()

binary_cols = [
    "high_price",
    "highly_rated",
    "high_availability",
    "popular_listing",
    "close_to_center",
    "professional_host",
    "luxury_listing"
]

for col in binary_cols:
    basket[col] = df[col].astype(bool)

## Kodiranje kategorijskih atributa

Kategorijski atributi ne mogu se izravno koristiti u FP-Growth algoritmu.

Zbog toga se primjenjuje One-Hot Encoding kojim se svaka kategorija pretvara u zaseban binarni atribut.

In [11]:
room_dummies = pd.get_dummies(
    df["room_type"],
    prefix="room_type"
).astype(bool)

group_dummies = pd.get_dummies(
    df["neighbourhood_group"],
    prefix="neighbourhood_group"
).astype(bool)

policy_dummies = pd.get_dummies(
    df["cancellation_policy"],
    prefix="cancellation_policy"
).astype(bool)

In [12]:
basket = pd.concat(
    [
        basket,
        room_dummies,
        group_dummies,
        policy_dummies
    ],
    axis=1
)

basket.head()

,high_price,highly_rated,high_availability,popular_listing,close_to_center,professional_host,luxury_listing,room_type_Entire home/apt,room_type_Hotel room,room_type_Private room,room_type_Shared room,neighbourhood_group_Bronx,neighbourhood_group_Brooklyn,neighbourhood_group_Manhattan,neighbourhood_group_Queens,neighbourhood_group_Staten Island,cancellation_policy_flexible,cancellation_policy_moderate,cancellation_policy_strict
0,True,True,True,False,False,True,False,False,False,True,False,False,True,False,False,False,False,False,True
1,False,True,True,True,True,False,False,True,False,False,False,False,False,True,False,False,False,True,False
2,False,True,True,True,False,False,False,True,False,False,False,False,True,False,False,False,False,True,False
3,False,True,True,True,False,False,False,False,False,True,False,False,True,False,False,False,False,True,False
4,False,False,False,True,True,False,False,True,False,False,False,False,False,True,False,False,True,False,False


In [13]:
basket.shape

(51045, 19)

## Pronalaženje frekventnih skupova

Primjenom FP-Growth algoritma pronalaze se skupovi obilježja koji se često pojavljuju zajedno.

Kao mjera učestalosti koristi se support, odnosno udio zapisa u kojima se određeni skup pojavljuje.

In [14]:
frequent_itemsets = fpgrowth(
    basket,
    min_support=0.08,
    use_colnames=True
)

frequent_itemsets = frequent_itemsets.sort_values(
    by="support",
    ascending=False
)

frequent_itemsets.head(20)

,support,itemsets
7,0.530532,frozenset({room_type_Entire home/apt})
8,0.499990,frozenset({close_to_center})
0,0.497522,frozenset({high_price})
9,0.491272,frozenset({popular_listing})
1,0.486022,frozenset({high_availability})
2,0.461514,frozenset({highly_rated})
3,0.445744,frozenset({room_type_Private room})
10,0.436066,frozenset({neighbourhood_group_Manhattan})
4,0.398296,frozenset({neighbourhood_group_Brooklyn})
115,0.373298,"frozenset({neighbourhood_group_Manhattan, clos..."


## Generiranje asocijacijskih pravila

Na temelju pronađenih frekventnih skupova generiraju se asocijacijska pravila.

Za procjenu kvalitete pravila koriste se sljedeće mjere:

- Support – učestalost pojavljivanja pravila u skupu podataka
- Confidence – vjerojatnost pojavljivanja posljedice ako je zadovoljen uvjet


In [15]:
rules = association_rules(
    frequent_itemsets,
    metric="confidence",
    min_threshold=0.6
)

rules = rules.sort_values(
    by="confidence",
    ascending=False
)

rules[
    [
        "antecedents",
        "consequents",
        "support",
        "confidence"
    ]
].head(20)

,antecedents,consequents,support,confidence
93,"frozenset({neighbourhood_group_Manhattan, prof...",frozenset({close_to_center}),0.086316,0.957202
45,"frozenset({high_availability, neighbourhood_gr...",frozenset({close_to_center}),0.125203,0.923688
54,"frozenset({room_type_Entire home/apt, neighbou...",frozenset({close_to_center}),0.120952,0.913043
6,"frozenset({room_type_Entire home/apt, neighbou...",frozenset({close_to_center}),0.242707,0.912835
117,"frozenset({room_type_Entire home/apt, cancella...",frozenset({close_to_center}),0.080987,0.912381
114,"frozenset({room_type_Entire home/apt, cancella...",frozenset({close_to_center}),0.081967,0.912342
64,"frozenset({room_type_Entire home/apt, neighbou...",frozenset({close_to_center}),0.111314,0.909702
106,"frozenset({room_type_Entire home/apt, neighbou...",frozenset({close_to_center}),0.082437,0.901264
67,"frozenset({room_type_Entire home/apt, neighbou...",frozenset({close_to_center}),0.107533,0.899984
94,"frozenset({close_to_center, professional_host})",frozenset({neighbourhood_group_Manhattan}),0.086316,0.895893


## Izdvajanje najzanimljivijih pravila

Kako bi se analizirala samo relevantna pravila, zadržavaju se pravila s visokim confidence vrijednostima.

In [16]:
interesting_rules = rules[
    (rules["confidence"] >= 0.7)
].sort_values(
    by="confidence",
    ascending=False
)

interesting_rules[
    [
        "antecedents",
        "consequents",
        "support",
        "confidence"
    ]
].head(15)

,antecedents,consequents,support,confidence
93,"frozenset({neighbourhood_group_Manhattan, prof...",frozenset({close_to_center}),0.086316,0.957202
45,"frozenset({high_availability, neighbourhood_gr...",frozenset({close_to_center}),0.125203,0.923688
54,"frozenset({room_type_Entire home/apt, neighbou...",frozenset({close_to_center}),0.120952,0.913043
6,"frozenset({room_type_Entire home/apt, neighbou...",frozenset({close_to_center}),0.242707,0.912835
117,"frozenset({room_type_Entire home/apt, cancella...",frozenset({close_to_center}),0.080987,0.912381
114,"frozenset({room_type_Entire home/apt, cancella...",frozenset({close_to_center}),0.081967,0.912342
64,"frozenset({room_type_Entire home/apt, neighbou...",frozenset({close_to_center}),0.111314,0.909702
106,"frozenset({room_type_Entire home/apt, neighbou...",frozenset({close_to_center}),0.082437,0.901264
67,"frozenset({room_type_Entire home/apt, neighbou...",frozenset({close_to_center}),0.107533,0.899984
94,"frozenset({close_to_center, professional_host})",frozenset({neighbourhood_group_Manhattan}),0.086316,0.895893


## Pravila povezana s visokom cijenom

Posebna pažnja posvećena je pravilima koja imaju visoku cijenu smještaja kao posljedicu.

Cilj je identificirati karakteristike koje se najčešće povezuju sa skupljim Airbnb smještajima.

In [17]:
high_price_rules = rules[
    rules["consequents"].apply(
        lambda x: "high_price" in x
    )
].sort_values(
    by="confidence",
    ascending=False
)

high_price_rules[
    [
        "antecedents",
        "consequents",
        "support",
        "confidence"
    ]
].head(15)

,antecedents,consequents,support,confidence


## Pravila povezana s visokom ocjenom

Analiziraju se obrasci koji se pojavljuju kod smještaja s visokim korisničkim ocjenama.

In [18]:
rating_rules = rules[
    rules["consequents"].apply(
        lambda x: "highly_rated" in x
    )
].sort_values(
    by="confidence",
    ascending=False
)

rating_rules[
    [
        "antecedents",
        "consequents",
        "support",
        "confidence"
    ]
].head(15)

,antecedents,consequents,support,confidence
108,"frozenset({room_type_Entire home/apt, neighbou...",frozenset({highly_rated}),0.082437,0.766624
81,"frozenset({room_type_Entire home/apt, neighbou...",frozenset({highly_rated}),0.091468,0.765535
70,"frozenset({room_type_Entire home/apt, close_to...",frozenset({highly_rated}),0.104418,0.762627
97,"frozenset({close_to_center, popular_listing, h...",frozenset({highly_rated}),0.084945,0.742212
21,"frozenset({close_to_center, popular_listing})",frozenset({highly_rated}),0.171555,0.742119
36,"frozenset({neighbourhood_group_Manhattan, clos...",frozenset({highly_rated}),0.128181,0.740661
29,"frozenset({neighbourhood_group_Manhattan, popu...",frozenset({highly_rated}),0.150691,0.738692
62,"frozenset({popular_listing, cancellation_polic...",frozenset({highly_rated}),0.117465,0.726524
79,"frozenset({room_type_Entire home/apt, popular_...",frozenset({highly_rated}),0.096033,0.724719
9,"frozenset({room_type_Entire home/apt, popular_...",frozenset({highly_rated}),0.190401,0.723571


## Pravila povezana s popularnošću smještaja

Ispituju se obilježja koja su karakteristična za popularne Airbnb oglase.

In [19]:
popular_rules = rules[
    rules["consequents"].apply(
        lambda x: "popular_listing" in x
    )
].sort_values(
    by="confidence",
    ascending=False
)

popular_rules[
    [
        "antecedents",
        "consequents",
        "support",
        "confidence"
    ]
].head(15)

,antecedents,consequents,support,confidence
77,"frozenset({high_availability, room_type_Entire...",frozenset({popular_listing}),0.099402,0.802721
88,"frozenset({high_availability, highly_rated, hi...",frozenset({popular_listing}),0.089333,0.801547
16,"frozenset({high_availability, highly_rated})",frozenset({popular_listing}),0.180351,0.799965
80,"frozenset({room_type_Entire home/apt, highly_r...",frozenset({popular_listing}),0.096033,0.782317
10,"frozenset({room_type_Entire home/apt, highly_r...",frozenset({popular_listing}),0.190401,0.776836
18,"frozenset({highly_rated, high_price})",frozenset({popular_listing}),0.177314,0.771809
58,"frozenset({cancellation_policy_flexible, highl...",frozenset({popular_listing}),0.119267,0.771316
32,"frozenset({neighbourhood_group_Brooklyn, highl...",frozenset({popular_listing}),0.141032,0.769124
3,frozenset({highly_rated}),frozenset({popular_listing}),0.354765,0.768699
63,"frozenset({highly_rated, cancellation_policy_s...",frozenset({popular_listing}),0.117465,0.768521


In [20]:
frequent_itemsets.head(20)

,support,itemsets
7,0.530532,frozenset({room_type_Entire home/apt})
8,0.499990,frozenset({close_to_center})
0,0.497522,frozenset({high_price})
9,0.491272,frozenset({popular_listing})
1,0.486022,frozenset({high_availability})
2,0.461514,frozenset({highly_rated})
3,0.445744,frozenset({room_type_Private room})
10,0.436066,frozenset({neighbourhood_group_Manhattan})
4,0.398296,frozenset({neighbourhood_group_Brooklyn})
115,0.373298,"frozenset({neighbourhood_group_Manhattan, clos..."


In [21]:
rules[["antecedents", "consequents", "support", "confidence"]].head(20)

,antecedents,consequents,support,confidence
93,"frozenset({neighbourhood_group_Manhattan, prof...",frozenset({close_to_center}),0.086316,0.957202
45,"frozenset({high_availability, neighbourhood_gr...",frozenset({close_to_center}),0.125203,0.923688
54,"frozenset({room_type_Entire home/apt, neighbou...",frozenset({close_to_center}),0.120952,0.913043
6,"frozenset({room_type_Entire home/apt, neighbou...",frozenset({close_to_center}),0.242707,0.912835
117,"frozenset({room_type_Entire home/apt, cancella...",frozenset({close_to_center}),0.080987,0.912381
114,"frozenset({room_type_Entire home/apt, cancella...",frozenset({close_to_center}),0.081967,0.912342
64,"frozenset({room_type_Entire home/apt, neighbou...",frozenset({close_to_center}),0.111314,0.909702
106,"frozenset({room_type_Entire home/apt, neighbou...",frozenset({close_to_center}),0.082437,0.901264
67,"frozenset({room_type_Entire home/apt, neighbou...",frozenset({close_to_center}),0.107533,0.899984
94,"frozenset({close_to_center, professional_host})",frozenset({neighbourhood_group_Manhattan}),0.086316,0.895893
